# Memory VS1 — prepared, no target result

Actual native-MLP saved-state compression with a new zero-storage shared-version witness and timeout-surviving activation records. Tied embedding/head storage remains **BLOCKED**; it is not silently installed. This is a distinct safety/evidence revision of historical trial005, not a claim that its timeout passed.

**CPU/static validation only.** Target preflight, full paired T4 training, evaluation, external peak and useful own Triton share are unexecuted. Own share may be zero or below 5%; component only, score null, admission unknown. See README and SOURCE_AUDIT for evidence limits.

Run only in an explicitly authorized, already prepared T4 session. No package installation, downloads, remote writes or queue submission occur. Canonical nonblocking GPU0 lock covers the complete workflow. All measured work is delegated to the untouched official harness.


In [ ]:
# Set existing local paths. A fresh notebook kernel is recommended.
AUTHORIZE_GPU = False  # Do not enable without explicit target-run authority.
OFFICIAL_REPO = "/kaggle/working/soup-kernel-challenge"
HF_HUB_CACHE = "/root/.cache/huggingface/hub"
OUTPUT_ROOT = "/kaggle/working/soup_candidates"
# Fixed cohort policy: equal 900 s native budget, seed 1, 64 steps,
# two repeats in base, solution, solution, base order.


## Frozen embedded source
The following cell embeds exactly the files in `candidate.zip` and their manifest. Only `solution/` is replaced in a fresh archived official checkout. Official harness/task/tests and fixed public data are hash-checked. Public cached model revision: `989aa7980e4cf806f80c7fef2b1adb7bc71aa306`.


In [ ]:
import json
SOURCES = {'solution/__init__.py': '"""Memory VS1: native MLP producer codec with durable zero-storage versions.\n\nComponent candidate only. No tied-table mutation or head math replacement.\nNo measured runtime, memory, quality, share or admission is claimed.\n"""\ndef apply():\n    from .adapter import install\n    result = install(mode="production", first_order_only=True)\n    # Native fallback is allowed; its lack of activation must remain visible.\n    import json\n    print("SOUP_MEMORY_VS1_INSTALL " + json.dumps(result, sort_keys=True), flush=True)\n    return result\n', 'solution/adapter.py': '"""Memory VS1: original revision of pinned native producer codec.\n\nNew: zero-storage shared-version witnesses replace weak-only wide checks;\nflushed cumulative activation checkpoints survive process termination.\n\nOnly two named native SavedTensor objects may receive hooks. There is no broad\nsaved_tensors_hooks context, value scan, CPU synchronization, custom derivative,\nshared input cast, alternate GEMM, or model/task-dependent measurement routing.\nProduction and explicit CPU-equivalent fixture modes are separate contracts.\n"""\nimport contextvars\nimport atexit\nimport functools\nimport hashlib\nimport inspect\nimport json\nimport weakref\nfrom dataclasses import dataclass\nfrom importlib.metadata import version\n\nimport torch\n\nfrom .restore import restore_half\nfrom .version_sentinel import make_witness, runtime_contract, UnsupportedVersionWitness\n\n\nEXPECTED = {\n    "bnb_matmul": "a219d40d62264de60a424e8f9f1bab7656136eb7264a575d39d57a7e2f25ed4a",\n    "linear_forward": "d52cf63457717cc718ca0a449dad0967171e679be90d5d2adcafe2e8a141093d",\n    "mlp_forward": "03df2d7685cf8446594448cbd33ef676a2b088779d99d9803abe3bed2ec7ad9b",\n    "silu_forward": "f9eaaa437c9f969e5f5c6f347b85a09fefcbae8824433dff69e0e83faa098863",\n}\nCONFLICT = "Calling register_hooks on a saved tensor whose hooks have already been set. Hint: only one pair of hooks is allowed at a time."\n_ACTIVE = contextvars.ContextVar("soup_mem001_invocation", default=None)\n_STATE = None\n_STATS = {}\n_SUMMARY_REGISTERED = False\n\n\ndef _count(name, value=1):\n    _STATS[name] = _STATS.get(name, 0) + value\n    count = _STATS[name]\n    # Geometric checkpoints keep stdout small and survive SIGKILL. Counts are\n    # lower bounds until the final record; they are NOT CUDA timing or steps.\n    if (_STATE is not None and _STATE["mode"] == "production"\n            and name in ("encoded_operands", "restores", "observed_mlp_calls")\n            and count > 0 and count & (count - 1) == 0):\n        print("SOUP_MEMORY_VS1_PROGRESS " + json.dumps(\n            {"event": name, "cumulative_lower_bound": snapshot(),\n             "completed_training": False}, sort_keys=True), flush=True)\n\n\ndef snapshot():\n    return dict(_STATS)\n\n\ndef reset_stats():\n    _STATS.clear()\n\n\ndef _hash(function):\n    # inspect.getsource unwraps decorated callables; a foreign/hub wrapper\n    # must not impersonate the pinned native implementation through metadata.\n    if hasattr(function, "__wrapped__"):\n        raise TypeError("Preexisting wrapped callable is outside the native contract")\n    return hashlib.sha256(inspect.getsource(function).encode()).hexdigest()\n\n\ndef _exit_summary():\n    print("SOUP_MEMORY_VS1_RUNTIME " + json.dumps(snapshot(), sort_keys=True), flush=True)\n\n\ndef _foreign_hooks(module, binding=None):\n    own = set() if binding is None else binding.handle_ids\n    for name in ("_forward_hooks", "_forward_pre_hooks", "_backward_hooks", "_backward_pre_hooks"):\n        if any(key not in own for key in getattr(module, name, {})):\n            return True\n    return False\n\n\ndef _global_hooks():\n    module = torch.nn.modules.module\n    return any(bool(getattr(module, name, {})) for name in (\n        "_global_forward_hooks", "_global_forward_pre_hooks",\n        "_global_backward_hooks", "_global_backward_pre_hooks"))\n\n\ndef _module_reason(mlp, x, state, binding=None):\n    if type(mlp) is not state["Qwen2MLP"] or type(x) is not torch.Tensor:\n        return "class"\n    if not state["first_order_only"]:\n        return "first_order_contract_absent"\n    if not mlp.training or not torch.is_grad_enabled() or torch.is_inference_mode_enabled() or not x.requires_grad:\n        return "not_first_order_training"\n    if x.dtype != torch.float32 or x.ndim not in (2, 3) or not x.is_contiguous() or x.numel() == 0:\n        return "input_metadata"\n    if torch.autograd.forward_ad.unpack_dual(x).tangent is not None:\n        return "forward_ad"\n    if type(mlp.act_fn) is not state["SiLUActivation"]:\n        return "activation"\n    if ("forward" in mlp.__dict__ or "forward" in mlp.act_fn.__dict__\n            or type(mlp.act_fn).forward is not state["silu_forward"]\n            or state["Linear4bit"].forward is not state["originals"]["linear_forward"]\n            or state["bnb"].matmul_4bit is not state["matmul_observer"]):\n        return "runtime_forward_identity"\n    if _global_hooks():\n        return "global_module_hooks"\n    modules = [mlp, mlp.act_fn, mlp.gate_proj, mlp.up_proj, mlp.down_proj]\n    if any(_foreign_hooks(module, binding) for module in modules):\n        return "foreign_module_hooks"\n    if state["mode"] == "production":\n        if x.device.type != "cuda" or torch.cuda.get_device_capability(x.device) != (7, 5):\n            return "not_t4"\n        if (mlp.hidden_size, mlp.intermediate_size) != (1536, 8960):\n            return "model_shape"\n        if torch.is_autocast_enabled("cuda") and torch.get_autocast_dtype("cuda") != torch.float16:\n            return "autocast_dtype"\n    elif x.device.type != "cpu" or torch.is_autocast_enabled("cpu"):\n        return "cpu_equivalent_device_or_autocast"\n    if x.shape[-1] != mlp.hidden_size:\n        return "hidden_shape"\n    for role, expected_shape in (("gate_proj", (mlp.intermediate_size, mlp.hidden_size)),\n                                 ("up_proj", (mlp.intermediate_size, mlp.hidden_size)),\n                                 ("down_proj", (mlp.hidden_size, mlp.intermediate_size))):\n        module = getattr(mlp, role)\n        if "forward" in module.__dict__:\n            return "instance_projection_forward"\n        if type(module) is not state["Linear4bit"] or module.bias is not None or not module.training:\n            return "projection_class_bias_or_mode"\n        if module.compute_dtype != torch.float16 or not module.compute_type_is_set:\n            return "projection_compute_contract"\n        if module.weight.requires_grad or module.weight.dtype != torch.uint8:\n            return "projection_weight"\n        qs = getattr(module.weight, "quant_state", None)\n        if (qs is None or tuple(qs.shape) != expected_shape or qs.quant_type != "nf4"\n                or qs.dtype != torch.float16 or not qs.nested or qs.blocksize != 64\n                or qs.state2.blocksize != 256 or getattr(qs, "packing_format_for_cpu", False)):\n            return "quantization_contract"\n    return None\n\n\nclass _Binding:\n    def __init__(self, mlp):\n        self.mlp = weakref.ref(mlp)\n        self.modules = {name: weakref.ref(getattr(mlp, name)) for name in ("gate_proj", "up_proj", "down_proj", "act_fn")}\n        self.handles = []\n        self.handle_ids = set()\n        self._add(mlp.gate_proj.register_forward_hook(self._wide))\n        self._add(mlp.up_proj.register_forward_hook(self._wide))\n        self._add(mlp.act_fn.register_forward_hook(self._activation))\n        self._add(mlp.down_proj.register_forward_pre_hook(self._product))\n\n    def _add(self, handle):\n        self.handles.append(handle)\n        self.handle_ids.add(handle.id)\n\n    def remove(self):\n        for handle in self.handles:\n            handle.remove()\n\n    def _invocation(self):\n        current = _ACTIVE.get()\n        return current if current is not None and current["binding"] is self else None\n\n    def _wide(self, module, inputs, output):\n        current = self._invocation()\n        if current is None:\n            return\n        role = "gate" if module is self.modules["gate_proj"]() else "up"\n        current[role + "_wide"] = output\n\n    def _activation(self, module, inputs, output):\n        current = self._invocation()\n        if current is not None and type(output) is torch.Tensor:\n            current["silu_node"] = output.grad_fn\n\n    def _product(self, module, inputs):\n        current = self._invocation()\n        if current is not None and len(inputs) == 1 and type(inputs[0]) is torch.Tensor:\n            current["mul_node"] = inputs[0].grad_fn\n\n\n@dataclass\nclass _Packet:\n    payload: torch.Tensor\n    version: int\n    wide_witness: torch.Tensor\n    wide_version: int\n    encoded: bool\n\n\ndef _unpack(packet):\n    if torch.is_grad_enabled():\n        raise RuntimeError("MEM001 first-order contract: opt out before forward for higher-order AD or saved-state inspection with grad enabled")\n    if packet.payload._version != packet.version:\n        raise RuntimeError("MEM001 saved producer payload was modified in-place")\n    if (packet.wide_witness._version != packet.wide_version\n            or packet.wide_witness.untyped_storage().nbytes() != 0):\n        raise RuntimeError("MEM001 widened operand version witness changed")\n    if packet.encoded:\n        restored = restore_half(packet.payload)\n        # Count only after the native CPU operation / CUDA launch returns.\n        # A CUDA return is dispatch evidence, not a synchronization proof.\n        _count("restores")\n        return restored\n    return packet.payload\n\n\ndef _register_one(raw_saved, wide, half):\n    # Witness holds a version counter but ZERO bytes of the FP32 storage.\n    # Never capture the invocation, modules, output or graph nodes.\n    try:\n        witness = make_witness(wide)\n    except UnsupportedVersionWitness:\n        _count("version_witness_declined")\n        return False\n    payload = half.detach()\n    packet = _Packet(payload, payload._version, witness, wide._version, True)\n    meta = (wide.data_ptr(), tuple(wide.shape), tuple(wide.stride()), wide.storage_offset(), wide.dtype, wide.device)\n    entered = False\n\n    def pack(tensor):\n        nonlocal entered\n        entered = True\n        actual = (tensor.data_ptr(), tuple(tensor.shape), tuple(tensor.stride()), tensor.storage_offset(), tensor.dtype, tensor.device)\n        if actual != meta or tensor._version != packet.wide_version:\n            _count("saved_operand_identity_fallback")\n            original = tensor.detach()\n            return _Packet(original, original._version, make_witness(tensor), tensor._version, False)\n        _count("encoded_operands")\n        _count("logical_half_payload_bytes", payload.numel() * payload.element_size())\n        return packet\n\n    try:\n        raw_saved.register_hooks(pack, _unpack)\n    except RuntimeError as exc:\n        if not entered and str(exc) == CONFLICT:\n            _count("existing_saved_hooks_declined")\n            return False\n        raise\n    return True\n\n\ndef _finish(current):\n    pairs = []\n    for role in ("gate", "up"):\n        half, wide = current.get(role + "_half"), current.get(role + "_wide")\n        if (current.get(role + "_calls") != 1 or type(half) is not torch.Tensor or type(wide) is not torch.Tensor\n                or half.dtype != torch.float16 or wide.dtype != torch.float32 or half.shape != wide.shape\n                or not half.is_contiguous() or not wide.is_contiguous() or half.device != wide.device\n                or half.grad_fn is None or wide.grad_fn is None or wide.grad_fn.name() != "ToCopyBackward0"\n                or wide.grad_fn.next_functions[0][0] is not half.grad_fn):\n            _count("producer_provenance_fallback")\n            return\n        pairs.append((half, wide))\n    silu, mul = current.get("silu_node"), current.get("mul_node")\n    if (silu is None or mul is None or silu.name() != "SiluBackward0" or mul.name() != "MulBackward0"\n            or not hasattr(silu, "_raw_saved_self") or not hasattr(mul, "_raw_saved_other")\n            or silu.next_functions[0][0] is not pairs[0][1].grad_fn\n            or mul.next_functions[0][0] is not silu or mul.next_functions[1][0] is not pairs[1][1].grad_fn):\n        _count("native_graph_fallback")\n        return\n    # Existing hooks are respected independently. No pair-atomicity claim.\n    _register_one(silu._raw_saved_self, pairs[0][1], pairs[0][0])\n    _register_one(mul._raw_saved_other, pairs[1][1], pairs[1][0])\n\n\ndef install(*, mode="production", first_order_only=True):\n    global _STATE, _SUMMARY_REGISTERED\n    if mode not in ("production", "cpu_equivalent"):\n        raise ValueError("Explicit production or cpu_equivalent mode required")\n    if _STATE is not None:\n        if (_STATE["bnb"].matmul_4bit is not _STATE["matmul_observer"]\n                or _STATE["Qwen2MLP"].forward is not _STATE["mlp_observer"]):\n            return {"installed": False, "reason": "foreign_postpatch"}\n        if _STATE["mode"] == mode and _STATE["first_order_only"] == first_order_only:\n            return {"installed": True, "reason": "already_installed", "mode": mode}\n        return {"installed": False, "reason": "different_installation_contract"}\n    import bitsandbytes as bnb\n    from transformers.activations import SiLUActivation\n    from transformers.models.qwen2.modeling_qwen2 import Qwen2MLP\n    if version("bitsandbytes") != "0.50.2" or version("transformers") != "5.16.1":\n        return {"installed": False, "reason": "package_version"}\n    if mode == "production" and (torch.__version__.split("+")[0] != "2.11.0" or version("triton") != "3.6.0"):\n        return {"installed": False, "reason": "target_torch_triton_version"}\n    originals = {"bnb_matmul": bnb.matmul_4bit, "linear_forward": bnb.nn.Linear4bit.forward,\n                 "mlp_forward": Qwen2MLP.forward, "silu_forward": SiLUActivation.forward}\n    try:\n        if any(_hash(function) != EXPECTED[name] for name, function in originals.items()):\n            return {"installed": False, "reason": "source_identity"}\n    except (OSError, TypeError):\n        return {"installed": False, "reason": "source_unavailable"}\n    try:\n        version_probe = runtime_contract("cpu")\n    except UnsupportedVersionWitness as exc:\n        return {"installed": False, "reason": "version_witness_contract", "detail": str(exc)}\n    state = {"mode": mode, "first_order_only": first_order_only, "bnb": bnb,\n             "Qwen2MLP": Qwen2MLP, "Linear4bit": bnb.nn.Linear4bit,\n             "originals": originals, "bindings": weakref.WeakKeyDictionary(),\n             "silu_forward": SiLUActivation.forward, "SiLUActivation": SiLUActivation}\n\n    @functools.wraps(originals["bnb_matmul"])\n    def matmul_observer(*args, **kwargs):\n        output = originals["bnb_matmul"](*args, **kwargs)\n        current = _ACTIVE.get()\n        if current is not None and len(args) >= 2:\n            a, weight = args[:2]\n            for role in ("gate", "up"):\n                module = current["binding"].modules[role + "_proj"]()\n                if module is not None and weight is module.weight:\n                    current[role + "_calls"] = current.get(role + "_calls", 0) + 1\n                    qs = kwargs.get("quant_state", args[2] if len(args) > 2 else None)\n                    bias = kwargs.get("bias", args[4] if len(args) > 4 else None)\n                    out = kwargs.get("out", args[3] if len(args) > 3 else None)\n                    if a.dtype == torch.float16 and qs is module.weight.quant_state and bias is None and out is None:\n                        current[role + "_half"] = output\n        return output\n\n    @functools.wraps(originals["mlp_forward"])\n    def mlp_observer(mlp, x):\n        if _ACTIVE.get() is not None:\n            return originals["mlp_forward"](mlp, x)\n        binding = state["bindings"].get(mlp)\n        reason = _module_reason(mlp, x, state, binding)\n        if reason is not None:\n            _count("fallback_" + reason)\n            return originals["mlp_forward"](mlp, x)\n        if binding is None:\n            binding = _Binding(mlp)\n            state["bindings"][mlp] = binding\n        current = {"binding": binding}\n        token = _ACTIVE.set(current)\n        try:\n            output = originals["mlp_forward"](mlp, x)\n            _finish(current)\n            _count("observed_mlp_calls")\n            return output\n        finally:\n            _ACTIVE.reset(token)\n            # All wide/capture/node references expire at this boundary.\n            current.clear()\n\n    state["matmul_observer"] = matmul_observer\n    state["mlp_observer"] = mlp_observer\n    bnb.matmul_4bit = matmul_observer\n    Qwen2MLP.forward = mlp_observer\n    _STATE = state\n    if mode == "production" and not _SUMMARY_REGISTERED:\n        atexit.register(_exit_summary)\n        _SUMMARY_REGISTERED = True\n    return {"installed": True, "mode": mode, "source_hashes": dict(EXPECTED), "first_order_only": first_order_only, "version_witness_probe": version_probe}\n\n\ndef uninstall():\n    global _STATE\n    if _STATE is None:\n        return\n    if _ACTIVE.get() is not None:\n        raise RuntimeError("Cannot uninstall an active MEM001 invocation")\n    state = _STATE\n    if state["bnb"].matmul_4bit is not state["matmul_observer"] or state["Qwen2MLP"].forward is not state["mlp_observer"]:\n        raise RuntimeError("MEM001 cannot overwrite a later foreign patch")\n    for binding in list(state["bindings"].values()):\n        binding.remove()\n    state["bnb"].matmul_4bit = state["originals"]["bnb_matmul"]\n    state["Qwen2MLP"].forward = state["originals"]["mlp_forward"]\n    _STATE = None\n', 'solution/fixture_common.py': '"""Original tiny-module/future-T4 fixture helpers; never used by apply()."""\nimport torch\nimport bitsandbytes as bnb\nfrom transformers.models.qwen2.configuration_qwen2 import Qwen2Config\nfrom transformers.models.qwen2.modeling_qwen2 import Qwen2MLP\n\n\ndef make_mlp(hidden=64, intermediate=128, device="cpu"):\n    config = Qwen2Config(hidden_size=hidden, intermediate_size=intermediate,\n                        num_attention_heads=4, num_key_value_heads=2,\n                        num_hidden_layers=1, hidden_act="silu")\n    mlp = Qwen2MLP(config)\n    generator = torch.Generator(device="cpu").manual_seed(305001)\n    for name, n, k in (("gate_proj", intermediate, hidden),\n                       ("up_proj", intermediate, hidden),\n                       ("down_proj", hidden, intermediate)):\n        layer = bnb.nn.Linear4bit(k, n, bias=False, compute_dtype=torch.float16,\n                                 compress_statistics=True, quant_type="nf4")\n        dense = (torch.randn((n, k), generator=generator) * 0.025).half()\n        layer.weight = bnb.nn.Params4bit(dense, requires_grad=False,\n                                         compress_statistics=True, quant_type="nf4")\n        layer = layer.to(device)\n        setattr(mlp, name, layer)\n    return mlp.train()\n\n\ndef compare(actual, expected):\n    assert actual.shape == expected.shape and actual.dtype == expected.dtype\n    masks = {"nan": torch.isnan, "posinf": torch.isposinf, "neginf": torch.isneginf}\n    for name, function in masks.items():\n        assert torch.equal(function(actual), function(expected)), name + " classification"\n    finite = torch.isfinite(actual) & torch.isfinite(expected)\n    dtype = {torch.float16: torch.int16, torch.float32: torch.int32}[actual.dtype]\n    assert torch.equal(actual[finite].contiguous().view(dtype), expected[finite].contiguous().view(dtype)), "finite bits/zero signs"\n    return {"finite_elements": int(finite.sum()), "nan_elements": int(torch.isnan(actual).sum()),\n            "inf_elements": int(torch.isinf(actual).sum()), "finite_bit_mismatches": 0}\n\n\ndef graph_signature(tensor):\n    nodes, seen = [], {}\n    def visit(node):\n        if node is None:\n            return None\n        if node in seen:\n            return seen[node]\n        index = len(nodes)\n        seen[node] = index\n        nodes.append(None)\n        nodes[index] = [node.name(), [visit(child) for child, _ in node.next_functions]]\n        return index\n    visit(tensor.grad_fn)\n    return nodes\n\n\ndef state_snapshot(mlp):\n    records = []\n    for name in ("gate_proj", "up_proj", "down_proj"):\n        layer = getattr(mlp, name)\n        qs = layer.weight.quant_state\n        tensors = [layer.weight, qs.absmax, qs.offset, qs.state2.absmax, qs.state2.code]\n        records.append((name, [(t.data_ptr(), t._version, t.detach().clone()) for t in tensors]))\n    return records\n\n\ndef check_state(mlp, before):\n    after = state_snapshot(mlp)\n    for (name1, a), (name2, b) in zip(before, after):\n        assert name1 == name2\n        for (ptr1, ver1, value1), (ptr2, ver2, value2) in zip(a, b):\n            assert ptr1 == ptr2 and ver1 == ver2 and torch.equal(value1, value2)\n\n\ndef run_one(mlp, value, cotangent, *, repeat=False, amp=False):\n    x = value.detach().clone().requires_grad_()\n    cpu_rng = torch.random.get_rng_state().clone()\n    gpu_rng = torch.cuda.get_rng_state(x.device).clone() if x.device.type == "cuda" else None\n    context = torch.autocast(x.device.type, dtype=torch.float16, enabled=amp)\n    with context:\n        output = mlp(x)\n    signature = graph_signature(output)\n    output.backward(cotangent, retain_graph=repeat)\n    if repeat:\n        output.backward(cotangent)\n    assert torch.equal(cpu_rng, torch.random.get_rng_state())\n    if gpu_rng is not None:\n        assert torch.equal(gpu_rng, torch.cuda.get_rng_state(x.device))\n    return output.detach(), x.grad.detach().clone(), signature\n', 'solution/restore.py': '"""Original useful widening only. No activation/derivative kernel is copied."""\nimport torch\n\n\ndef restore_half(payload):\n    if payload.device.type == "cpu":\n        return payload.to(torch.float32)\n    if payload.device.type != "cuda":\n        raise RuntimeError("MEM001 unsupported restore device")\n    from .restore_kernel import launch_restore\n    return launch_restore(payload)\n', 'solution/restore_kernel.py': '"""Original, unaudited-on-T4 half-to-FP32 restoration, not SwiGLU."""\nimport torch\nimport triton\nimport triton.language as tl\n\n\n@triton.jit\ndef soup_mem001_restore_half_to_fp32(source, destination, count,\n                                    BLOCK: tl.constexpr):\n    offset = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)\n    value = tl.load(source + offset, mask=offset < count, other=0)\n    tl.store(destination + offset, value.to(tl.float32), mask=offset < count)\n\n\ndef launch_restore(source):\n    if source.dtype != torch.float16 or not source.is_contiguous():\n        raise RuntimeError("MEM001 restore requires contiguous half payload")\n    destination = torch.empty(source.shape, dtype=torch.float32, device=source.device)\n    soup_mem001_restore_half_to_fp32[(triton.cdiv(source.numel(), 256),)](\n        source, destination, source.numel(), BLOCK=256, num_warps=4,\n        num_stages=1, enable_fp_fusion=False)\n    return destination\n', 'solution/verify_memory.py': '"""Future explicitly authorized T4 preflight. Not run by solution.apply().\n\nNo download, install, benchmark, tolerance search or harness modification.\nExact finite restored values/output/input gradients; nonfinite classes and\nsigned-zero identity. NaN payload bits are outside this declared gate.\n"""\nimport argparse\nimport hashlib\nimport json\nimport traceback\nfrom pathlib import Path\n\nimport torch\n\nfrom . import adapter\nfrom .fixture_common import make_mlp, compare, run_one, state_snapshot, check_state\nfrom .restore import restore_half\n\n\ndef environment_gate():\n    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:\n        raise RuntimeError("Preflight needs one visible, explicitly authorized CUDA GPU")\n    if torch.cuda.get_device_capability(0) != (7, 5):\n        raise RuntimeError("Preflight is fixed to T4/SM75")\n    installed = adapter.install(mode="production")\n    if not installed["installed"]:\n        raise RuntimeError("Production source/environment gate: " + repr(installed))\n    adapter.uninstall()\n    return {"torch": torch.__version__, "torch_git": torch.version.git_version,\n            "cuda_runtime": torch.version.cuda, "gpu_name": torch.cuda.get_device_name(0),\n            "capability": list(torch.cuda.get_device_capability(0)), "source_gate": installed}\n\n\ndef restore_gate():\n    # Every binary16 encoding, including zero/subnormals/Inf/NaNs. Finite bits\n    # must match native widening; NaN payload identity is deliberately excluded.\n    codes = torch.arange(65536, dtype=torch.int32).to(torch.int16).view(torch.float16).cuda()\n    rows = []\n    for length in (1, 7, 257, 65536, 65539):\n        source = codes[:length].contiguous() if length <= codes.numel() else torch.cat((codes, codes[:3]))\n        actual, expected = restore_half(source), source.float()\n        rows.append({"length": length, **compare(actual, expected)})\n        assert actual.stride() == expected.stride()\n    return {"binary16_restore_cases": rows, "nan_payload_bits": "not asserted"}\n\n\ndef compare_pair(mlp, x, dy, amp, repeat=False):\n    adapter.uninstall()\n    baseline = run_one(mlp, x, dy, amp=amp, repeat=repeat)\n    assert adapter.install(mode="production")["installed"]\n    adapter.reset_stats()\n    candidate = run_one(mlp, x, dy, amp=amp, repeat=repeat)\n    result = {"shape": list(x.shape), "amp_fp16": amp,\n              "output": compare(candidate[0], baseline[0]),\n              "input_gradient": compare(candidate[1], baseline[1]),\n              "stats": adapter.snapshot()}\n    assert candidate[0].stride() == baseline[0].stride()\n    assert candidate[1].stride() == baseline[1].stride()\n    assert candidate[2] == baseline[2], "native graph topology changed"\n    assert result["stats"].get("encoded_operands") == 2\n    assert result["stats"].get("restores") == (4 if repeat else 2)\n    adapter.uninstall()\n    return result\n\n\ndef actual_module_gate(mlp):\n    before = state_snapshot(mlp)\n    rows = []\n    for shape, amp, repeat in (((1, 3, 1536), False, True),\n                               ((1, 3, 1536), True, False),\n                               ((1, 356, 1536), True, False),\n                               ((1, 513, 1536), True, False)):\n        count = shape[0] * shape[1] * shape[2]\n        x = torch.linspace(-.2, .2, count, device="cuda").reshape(shape)\n        dy = torch.linspace(-.5, .5, count, device="cuda").reshape(shape)\n        rows.append(compare_pair(mlp, x, dy, amp, repeat))\n    check_state(mlp, before)\n    return {"fixed_shape_cases": rows, "weight_and_quantstate_unchanged": True}\n\n\ndef exceptional_gate(mlp):\n    rows = []\n    shape = (1, 3, 1536)\n    for kind in ("scaled_gradient", "input_inf", "input_nan", "cotangent_inf"):\n        x = torch.full(shape, .125, device="cuda")\n        dy = torch.ones_like(x)\n        if kind == "scaled_gradient":\n            dy.mul_(65536.)\n        elif kind == "input_inf":\n            x[0, 0, 0] = float("inf")\n        elif kind == "input_nan":\n            x[0, 0, 0] = float("nan")\n        else:\n            dy[0, 0, 0] = float("inf")\n        rows.append({"kind": kind, **compare_pair(mlp, x, dy, True)})\n    return rows\n\n\ndef scaler_optimizer_gate(mlp):\n    rows = []\n    initial = torch.linspace(-.1, .1, 3 * 1536, device="cuda").reshape(1, 3, 1536)\n    for enabled in (False, True):\n        adapter.uninstall()\n        if enabled:\n            assert adapter.install(mode="production")["installed"]\n        adapter.reset_stats()\n        x = torch.nn.Parameter(initial.clone())\n        optimizer = torch.optim.AdamW([x], lr=1e-4)\n        scaler = torch.amp.GradScaler("cuda", init_scale=1024.)\n        with torch.autocast("cuda", dtype=torch.float16):\n            # Real Qwen hidden states are ordinary Tensor values, not a\n            # Parameter subclass. Preserve the optimizer leaf and its native\n            # differentiable edge identically on baseline/candidate sides.\n            hidden = x.view_as(x)\n            y = mlp(hidden)\n            loss = y.square().mean()\n        scaler.scale(loss).backward()\n        scaler.unscale_(optimizer)\n        gradient = x.grad.detach().clone()\n        scaler.step(optimizer)\n        scaler.update()\n        rows.append((loss.detach(), gradient, x.detach().clone(), scaler.get_scale(), adapter.snapshot()))\n    for i in range(3):\n        compare(rows[0][i], rows[1][i])\n    assert rows[0][3] == rows[1][3]\n    assert rows[1][4].get("encoded_operands") == 2, {"candidate_stats": rows[1][4], "baseline_stats": rows[0][4]}\n    adapter.uninstall()\n    return {"native_AdamW_and_GradScaler_leaf_input_control": "exact", "scale": rows[1][3],\n            "not_claimed": "full q/v adapter optimizer trajectory"}\n\n\ndef saved_hook_fallback_gate(mlp):\n    assert adapter.install(mode="production")["installed"]\n    adapter.reset_stats()\n    counts = {"pack": 0, "unpack": 0}\n    def pack(t):\n        counts["pack"] += 1\n        return t.detach()\n    def unpack(t):\n        counts["unpack"] += 1\n        return t\n    x = torch.zeros((1, 3, 1536), device="cuda")\n    with torch.autograd.graph.saved_tensors_hooks(pack, unpack):\n        run_one(mlp, x, torch.ones_like(x), amp=True)\n    stats = adapter.snapshot()\n    assert stats.get("encoded_operands", 0) == 0\n    assert stats.get("existing_saved_hooks_declined") == 2\n    adapter.uninstall()\n    return {"counts": counts, "stats": stats}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--out", default="preflight_memory_vs1.json")\n    args = parser.parse_args()\n    output = Path(args.out)\n    result = {"candidate": "memory-vs1", "status": "RUNNING", "groups": [],\n              "gates": "exact finite bits/strides/gradients and nonfinite classifications; no relaxed tolerance",\n              "source_sha256": hashlib.sha256(Path(adapter.__file__).read_bytes()).hexdigest(),\n              "not_a_benchmark_or_admission": True}\n    def record(name, function):\n        details = function()\n        torch.cuda.synchronize()\n        result["groups"].append({"name": name, "status": "PASS", "details": details})\n        output.write_text(json.dumps(result, indent=2) + "\\n")\n    active_name = "environment"\n    try:\n        record(active_name, environment_gate)\n        from .version_sentinel import runtime_contract\n        active_name = "cuda_version_witness"\n        record(active_name, lambda: runtime_contract("cuda"))\n        active_name = "restore_binary16"\n        record(active_name, restore_gate)\n        mlp = make_mlp(1536, 8960, "cuda")\n        for name, function in (("actual_fixed_MLP", actual_module_gate),\n                               ("exceptional_classes", exceptional_gate),\n                               ("native_scaler_optimizer", scaler_optimizer_gate),\n                               ("existing_saved_hook_fallback", saved_hook_fallback_gate)):\n            active_name = name\n            record(name, lambda function=function: function(mlp))\n        result["status"] = "PASS"\n    except Exception as exc:\n        result["status"] = "FAIL"\n        result["groups"].append({"name": active_name, "status": "FAIL", "error": repr(exc), "traceback": traceback.format_exc()})\n    finally:\n        adapter.uninstall()\n        output.write_text(json.dumps(result, indent=2) + "\\n")\n    print(json.dumps({"status": result["status"], "groups": len(result["groups"])}), flush=True)\n    raise SystemExit(result["status"] != "PASS")\n\n\nif __name__ == "__main__":\n    main()\n', 'solution/version_sentinel.py': '"""Original zero-storage version witness for a native saved FP32 operand.\n\nA detached Tensor shares the input\'s version counter. Replacing ONLY that\nwitness\'s .data with an empty tensor discards its storage without modifying the\nsource\'s value, storage or version. This is not .data mutation of model values.\nThe private/version-sensitive mechanism is checked on the installed runtime;\nunsupported behavior declines the codec. .data mutations are outside native\nPyTorch version tracking too, and are not claimed detectable here.\n"""\nimport gc\nimport weakref\n\nimport torch\n\n\nclass UnsupportedVersionWitness(RuntimeError):\n    pass\n\n\ndef make_witness(source):\n    if type(source) is not torch.Tensor or source.layout != torch.strided:\n        raise UnsupportedVersionWitness("ordinary strided Tensor required")\n    before = (source.data_ptr(), source._version, tuple(source.shape),\n              tuple(source.stride()), source.dtype, source.device)\n    witness = source.detach()\n    # set_ would increment the SOURCE\'s shared version: do not use it here.\n    witness.data = torch.empty(0, dtype=source.dtype, device=source.device)\n    after = (source.data_ptr(), source._version, tuple(source.shape),\n             tuple(source.stride()), source.dtype, source.device)\n    if (before != after or witness._version != source._version\n            or witness.numel() != 0 or witness.untyped_storage().nbytes() != 0\n            or witness.grad_fn is not None or witness.requires_grad):\n        raise UnsupportedVersionWitness("runtime does not preserve a zero-storage shared version")\n    return witness\n\n\ndef runtime_contract(device="cpu"):\n    """Bounded synthetic check; consumes no RNG and changes no model tensor."""\n    wide = torch.arange(4, dtype=torch.float32, device=device)\n    alias = wide.detach()\n    ref = weakref.ref(wide)\n    witness = make_witness(wide)\n    expected = wide._version\n    del wide\n    gc.collect()\n    if ref() is not None:\n        raise UnsupportedVersionWitness("witness retained original Python Tensor")\n    alias.add_(1)\n    if witness._version != expected + 1:\n        raise UnsupportedVersionWitness("detached alias mutations are not witnessed")\n    del alias\n    if witness.untyped_storage().nbytes() != 0:\n        raise UnsupportedVersionWitness("witness retained nonzero storage")\n    return {"device": str(device), "wide_object_released": True,\n            "witness_storage_bytes": 0, "detached_alias_mutation_detected": True}\n'}
SOURCE_HASHES = {'solution/__init__.py': 'bd440ab58a4391f5311e2491e80e1661602e9d7ebee41069923fa5d6099605c6', 'solution/adapter.py': 'dee688af8fe8cba35757e7613cd1c2e9f7195a780463bee8e19b230abf03284a', 'solution/fixture_common.py': '3681aeaa6020ab722110ca23c952657d2e8103340b874dee00d90c1d4bad317e', 'solution/restore.py': 'ef7d677ef08e01123b360f7c503e80873a245d8901c9374b08fdd7ab090b1b87', 'solution/restore_kernel.py': '64811560821f78f1bfeadf2b0eeda55d8258fa4d8daed866e0ec727c9ea617ef', 'solution/verify_memory.py': '770aec249b0b9eb286f855955b384656da412595318dee7d3de7a8b451a207b4', 'solution/version_sentinel.py': '76dcaabc808f3b9c051f05c8705ccba66150b2a25e1649bcd87a9cc1addd13db'}
PROTECTED_HASHES = {'harness/_train.py': '45a62642dceab32cf3efcda504de87a9c25ea28b119b1806f56d14ff1146a89d', 'harness/check_rules.py': '879ab1861674f6e714eadd00aaf044235d98526dd75fab1035deec3401121ef5', 'harness/common.py': 'c83066f24f266dab62ebcc9c2d44f5a65902760b53da912e65ebfe511579c2db', 'harness/eval_adapter.py': '47cddbb8e4579c69f9535b099ff2371b1c3af08bfbd93d14bf79aadd74d38482', 'harness/example_reference.py': '6a11055f4a1822e53971c228c6e3e3aa6b88edd9eb6db81a388e26a70a030ceb', 'harness/run_task.py': 'd935a8959f348977313f2eef36870ff2699b9ba013a7978918ff35e148968a79', 'harness/score.py': '731c9b644dcc4069b0b069616f438064d1f7f376a202feb15201fd910ec688c5', 'task/prepare_data.py': 'a8543f11a55aff80dbdc714a0e62391244a5296616eea2e34dc2eb3ff3ec18f7', 'task/task.yaml': 'dd70c739fcd143e06aef3bb1efcc9386f682bf1de26f30e08409d86a97ff47ce', 'tests/conftest.py': 'a82eabab468ea0ef1e003cccd7c3827ce2f4d00d09c32948a34fb48221f7337a', 'tests/test_callback.py': '1919303a3670c0a33abbfce9b33eab0913bb64a3a21c3312d86531c32b5bf490', 'tests/test_config.py': 'db808c44470d144b7cf18b7329492f41f76e37cfc24306e7cba0488bca9d787c', 'tests/test_formats.py': '5d656ed50c59f794e934473d459ea85635f892ba028e192414474e9b5d65919e', 'tests/test_loader.py': '3dad1bea2d33aab97e52b7cd44888d768bc9519af44fbe6cf3e96c260026632f', 'tests/test_trainer_init.py': 'b7608d62a40983bf7c9e5b270e0e527b641a837e462d6df2909fec88f607e821'}
RUNNER_SOURCE = '"""Prepared-only notebook support: reviewed local assets, unchanged official harness.\n\nThis file is not run by preparation and is not part of the queue solution ZIP.\nGPU execution requires explicit future authorization and a prepared exact stack.\n"""\nimport argparse\nimport ctypes\nimport fcntl\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport tarfile\nimport tempfile\nimport time\nimport uuid\n\nCOMMIT = \'c7c2fcd15972ce8b898d8d3ff732ee192c817ceb\'\nMODEL = \'Qwen/Qwen2.5-1.5B-Instruct\'\nREVISION = \'989aa7980e4cf806f80c7fef2b1adb7bc71aa306\'\nPACKAGES = {\'torch\': \'2.11.0+cu128\', \'triton\': \'3.6.0\', \'soup-cli\': \'0.77.0\',\n            \'transformers\': \'5.16.1\', \'peft\': \'0.20.0\', \'trl\': \'0.29.1\',\n            \'bitsandbytes\': \'0.50.2\', \'accelerate\': \'1.14.0\', \'datasets\': \'4.8.5\'}\nDATA = {\'data/train.jsonl\': \'93d036d912bd81c337dc29893e13a76f36161d96f896ef4e02492e2f1a21a0dd\',\n        \'data/eval_public.jsonl\': \'b7f85b78ba3186dce4deab7cfb8bf8dcf7998e1517c5749ac0f5005886c6f70e\'}\nLOCK_PATH = Path(\'/tmp/soup-gpu0.lock\')\nBLOCK_PATH = Path(\'/tmp/soup-gpu0.lock.blocked\')\n_DEDICATED_RUNNER = False\nPREFIXES = (\'SOUP_MEMORY_VS1_INSTALL \', \'SOUP_MEMORY_VS1_PROGRESS \', \'SOUP_MEMORY_VS1_RUNTIME \')\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(4 * 1024 * 1024), b\'\'): h.update(block)\n    return h.hexdigest()\n\n\ndef dump(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\')\n\n\ndef require(condition, message):\n    if not condition: raise RuntimeError(message)\n\n\ndef public_cache_inventory(cache):\n    root = Path(cache).resolve() / \'models--Qwen--Qwen2.5-1.5B-Instruct\'\n    require((root / \'refs/main\').read_text().strip() == REVISION, \'Prepared model refs/main revision mismatch\')\n    snapshot = root / \'snapshots\' / REVISION\n    required = {\'config.json\', \'tokenizer_config.json\', \'tokenizer.json\'}\n    index = snapshot / \'model.safetensors.index.json\'\n    if index.is_file():\n        required.add(index.name)\n        required.update(json.loads(index.read_text())[\'weight_map\'].values())\n    else:\n        required.add(\'model.safetensors\')\n    require(all((snapshot / name).is_file() for name in required), \'Prepared model snapshot is incomplete; do not download inside run\')\n    hashes = {}\n    for name in sorted(required):\n        path = snapshot / name\n        require(path.resolve().is_relative_to(root.resolve()), \'Model snapshot link escapes selected public model\')\n        hashes[name] = sha(path)\n    return {\'model\': MODEL, \'revision\': REVISION, \'required_file_hashes\': hashes}\n\n\ndef stage(repo, output_root, sources, hashes, protected_hashes):\n    repo = Path(repo).resolve()\n    require(subprocess.check_output([\'git\', \'-C\', str(repo), \'rev-parse\', \'HEAD\'], text=True).strip() == COMMIT,\n            \'Prepared official checkout must be at the exact starter commit\')\n    for name, expected in protected_hashes.items():\n        require(sha(repo / name) == expected, \'Changed/missing official file: \' + name)\n    for name, expected in DATA.items():\n        require(sha(repo / name) == expected, \'Fixed public data hash mismatch: \' + name)\n    root = Path(output_root).resolve() / (\'memory_vs1_\' + time.strftime(\'%Y%m%dT%H%M%SZ\', time.gmtime()) + \'_\' + uuid.uuid4().hex[:8])\n    root.mkdir(parents=True, exist_ok=False)\n    target = root / \'repo\'; target.mkdir()\n    archive = subprocess.check_output([\'git\', \'-C\', str(repo), \'archive\', COMMIT])\n    archive_path = root / \'official.tar\'; archive_path.write_bytes(archive)\n    with tarfile.open(archive_path) as tar:\n        tar.extractall(target, filter=\'data\')\n    (target / \'data\').mkdir(exist_ok=True)\n    for name in DATA: shutil.copy2(repo / name, target / name)\n    shutil.rmtree(target / \'solution\')\n    (target / \'solution\').mkdir()\n    for name, source in sources.items():\n        require(name.startswith(\'solution/\') and \'..\' not in Path(name).parts, \'Unsafe embedded source name\')\n        raw = source.encode()\n        require(hashlib.sha256(raw).hexdigest() == hashes[name], \'Embedded candidate source mismatch: \' + name)\n        dest = target / name; dest.parent.mkdir(parents=True, exist_ok=True); dest.write_bytes(raw)\n    for name, expected in protected_hashes.items():\n        require(sha(target / name) == expected, \'Staged official file mismatch: \' + name)\n    dump(root / \'source_manifest.json\', hashes)\n    dump(root / \'official_manifest.json\', protected_hashes)\n    return root, target\n\n\ndef process_table():\n    """Linux process identity: PID, parent PID, kernel start ticks, state."""\n    table = {}\n    for entry in Path(\'/proc\').iterdir():\n        if not entry.name.isdigit(): continue\n        try:\n            fields = (entry/\'stat\').read_text().rsplit(\')\', 1)[1].split()\n            table[int(entry.name)] = {\'ppid\': int(fields[1]), \'birth\': int(fields[19]), \'state\': fields[0]}\n        except (FileNotFoundError, ProcessLookupError):\n            continue\n        except PermissionError:\n            # Unknown inaccessible processes are never adopted/signalled.\n            continue\n    return table\n\n\nclass OwnedProcesses:\n    """Track only descendants of our child, preserving PID birth identity."""\n    def __init__(self, process):\n        self.process = process\n        table = process_table()\n        if process is None:\n            self.owned = {pid: row[\'birth\'] for pid, row in table.items() if row[\'ppid\'] == os.getpid()}\n        else:\n            require(process.pid in table, \'Owned child identity unavailable; cannot safely supervise\')\n            self.owned = {process.pid: table[process.pid][\'birth\']}\n        self.observe()\n\n    def observe(self):\n        table = process_table()\n        # Existing owned children remain tracked when reparented or setsid().\n        while True:\n            live = {pid for pid, birth in self.owned.items()\n                    if pid in table and table[pid][\'birth\'] == birth}\n            new = {pid: row[\'birth\'] for pid, row in table.items()\n                   if (row[\'ppid\'] in live or (_DEDICATED_RUNNER and row[\'ppid\'] == os.getpid()))\n                   and pid not in self.owned}\n            if not new: break\n            self.owned.update(new)\n        return {pid: table[pid] for pid, birth in self.owned.items()\n                if pid in table and table[pid][\'birth\'] == birth and table[pid][\'state\'] != \'Z\'}\n\n    def signal_owned(self, signo):\n        for pid, row in self.observe().items():\n            # pidfd is the actual signal target: PID reuse after validation\n            # cannot redirect the signal to an unrelated process.\n            try:\n                descriptor = os.pidfd_open(pid, 0)\n            except ProcessLookupError:\n                continue\n            try:\n                current = process_table().get(pid)\n                if current and current[\'birth\'] == row[\'birth\']:\n                    signal.pidfd_send_signal(descriptor, signo, None, 0)\n            except ProcessLookupError:\n                pass\n            finally:\n                os.close(descriptor)\n\n    def cleanup(self):\n        handlers = {sig: signal.signal(sig, signal.SIG_IGN) for sig in (signal.SIGINT, signal.SIGTERM)}\n        try:\n            return self._cleanup()\n        finally:\n            for sig, handler in handlers.items(): signal.signal(sig, handler)\n\n    def _cleanup(self):\n        # Stop known owners before rescanning, so ordinary children cannot\n        # keep forking while their parent is being killed. setsid is irrelevant.\n        frozen = set()\n        while True:\n            self.signal_owned(signal.SIGSTOP)\n            now = set(self.observe())\n            if now <= frozen: break\n            frozen.update(now)\n        self.signal_owned(signal.SIGKILL)\n        if self.process is not None:\n            try: self.process.wait(timeout=5)\n            except subprocess.TimeoutExpired: pass\n        # Retain the lock until terminal, including an uninterruptible kernel.\n        # An owner can investigate the durable block marker; no blind retry.\n        while self.observe():\n            self.signal_owned(signal.SIGKILL)\n            time.sleep(.05)\n        if _DEDICATED_RUNNER:\n            while True:\n                try:\n                    pid, _ = os.waitpid(-1, os.WNOHANG)\n                    if not pid: break\n                except ChildProcessError: break\n        return {\'all_tracked_owned_processes_terminal\': not bool(self.observe()),\n                \'owned_pid_birth_ticks\': {str(pid): birth for pid, birth in self.owned.items()},\n                \'limit\': \'Escaped descendants between observations cannot be universally excluded; any interruption blocks lane for reconciliation\'}\n\n\ndef command(args, log, cwd, env):\n    # Native harness owns normal timeout semantics. We additionally supervise\n    # session-escaping descendants for notebook interrupt / wrapper failure.\n    start = time.perf_counter()\n    process = tracker = None\n    try:\n        with Path(log).open(\'x\') as stream:\n            process = subprocess.Popen(args, cwd=cwd, env=env, stdout=stream,\n                                       stderr=subprocess.STDOUT, start_new_session=True)\n            tracker = OwnedProcesses(process)\n            while True:\n                tracker.observe()\n                try:\n                    code = process.wait(timeout=.1)\n                    break\n                except subprocess.TimeoutExpired:\n                    continue\n            remaining = tracker.observe()\n            if remaining:\n                cleanup = tracker.cleanup()\n                dump(BLOCK_PATH, {\'reason\': \'owned descendants survived wrapper exit\', \'cleanup\': cleanup, \'log\': str(log)})\n                raise RuntimeError(\'Owned descendants survived command exit; GPU lane blocked for reconciliation\')\n    except BaseException as exc:\n        dump(BLOCK_PATH, {\'reason\': type(exc).__name__, \'cleanup\': \'IN_PROGRESS\', \'log\': str(log),\n                          \'manual_owner_reconciliation_required\': True})\n        cleanup = tracker.cleanup() if tracker is not None else {\'identity_tracking_available\': False}\n        # Even successful tracked cleanup cannot prove a short-lived owner\n        # never orphaned a descendant between observations. Do not auto-retry.\n        dump(BLOCK_PATH, {\'reason\': type(exc).__name__, \'cleanup\': cleanup, \'log\': str(log),\n                          \'manual_owner_reconciliation_required\': True})\n        dump(str(log)+\'.interrupted.json\', {\'status\': \'INTERRUPTED_OR_UNCERTAIN\', \'cleanup\': cleanup,\n                                            \'score\': None, \'admission\': \'UNKNOWN\'})\n        raise\n    return {\'command\': args, \'exit_code\': code,\n            \'whole_command_wall_s\': time.perf_counter() - start, \'log\': str(log)}\n\n\ndef summarize_evidence(result_path, root):\n    raw = json.loads(Path(result_path).read_text()) if Path(result_path).is_file() else {}\n    errors, routes = [], []\n    if raw.get(\'packages_changed\'): errors.append(\'packages changed\')\n    runs = raw.get(\'runs\', {})\n    for side in (\'base\', \'solution\'):\n        entries = runs.get(side, [])\n        if len(entries) != 2: errors.append(side + \': expected two fixed-task runs\')\n        for run in entries:\n            metrics = run.get(\'metrics\') or {}\n            required = (run.get(\'exit_code\') == 0, metrics.get(\'exit_code\') == 0,\n                        metrics.get(\'steps\') == 64, run.get(\'global_step\') == 64,\n                        bool(run.get(\'adapter_tensors\')), bool(run.get(\'adapter_config\')),\n                        isinstance(run.get(\'wall_s\'), (float, int)) and run.get(\'wall_s\', 0) > 0,\n                        isinstance(run.get(\'gpu_peak_mib\'), (float, int)) and run.get(\'gpu_peak_mib\', 0) > 0)\n            if not all(required): errors.append(side + \'/\' + run.get(\'run\', \'?\') + \': incomplete native run\')\n            if side == \'solution\':\n                log = Path(root) / \'runs\' / run[\'run\'] / \'train.log\'\n                events = []\n                if log.is_file():\n                    for line in log.read_text(errors=\'replace\').splitlines():\n                        for prefix in PREFIXES:\n                            if line.startswith(prefix):\n                                try: events.append({\'kind\': prefix.strip(), \'value\': json.loads(line[len(prefix):])})\n                                except ValueError: pass\n                routes.append({\'run\': run[\'run\'], \'events\': events,\n                               \'own_kernel_ms\': metrics.get(\'own_kernel_ms\'),\n                               \'timed_loop_s\': metrics.get(\'timed_loop_s\'),\n                               \'timed_steps\': metrics.get(\'timed_steps\')})\n                finals = [e[\'value\'] for e in events if e[\'kind\'] == \'SOUP_MEMORY_VS1_RUNTIME\']\n                if not finals or finals[-1].get(\'encoded_operands\', 0) <= 0 or finals[-1].get(\'restores\', 0) <= 0:\n                    errors.append(run[\'run\'] + \': final nonvacuous codec activation absent\')\n        quality = raw.get(\'evals\', {}).get(side, {})\n        value = quality.get(\'mean_loss\')\n        if quality.get(\'exit_code\') != 0 or not isinstance(value, (float, int)) or not math.isfinite(value):\n            errors.append(side + \': finite successful plain public evaluation absent\')\n    # A crash can leave flushed train.log events without any result.json.\n    recovered_logs = []\n    for log in sorted((Path(root) / \'runs\').glob(\'solution_*/train.log\')):\n        events = []\n        for line in log.read_text(errors=\'replace\').splitlines():\n            for prefix in PREFIXES:\n                if line.startswith(prefix):\n                    try: events.append({\'kind\': prefix.strip(), \'value\': json.loads(line[len(prefix):])})\n                    except ValueError: pass\n        recovered_logs.append({\'run\': log.parent.name, \'events\': events,\n                               \'partial_unless_complete_result\': True})\n    report = {\'candidate_id\': \'soup-memory-vs1-20261030\',\n              \'status\': \'COMPLETE_COMPONENT_MEASUREMENT\' if not errors else \'FAILED_OR_INCOMPLETE\',\n              \'errors\': errors, \'score\': None, \'admission\': \'UNKNOWN\',\n              \'standalone_eligibility\': \'Useful own Triton share must be independently measured; may be below 5%\',\n              \'routes\': routes, \'timeout_surviving_logs\': recovered_logs, \'raw_summary_untrusted_if_incomplete\': raw.get(\'summary\'),\n              \'ratios\': raw.get(\'summary\', {}).get(\'ratios\') if not errors else None}\n    dump(Path(root) / \'review_summary.json\', report)\n    return report\n\n\ndef create_active_marker(token):\n    # Exclusive create + fsync happens before any GPU-capable subprocess. A\n    # SIGKILL therefore leaves an owner-bound reconciliation marker.\n    with BLOCK_PATH.open(\'x\') as handle:\n        json.dump({\'owner_token\': token, \'phase\': \'active\', \'supervisor_pid\': os.getpid(),\n                   \'supervisor_birth_ticks\': process_table()[os.getpid()][\'birth\'],\n                   \'manual_owner_reconciliation_required_unless_cleanly_removed\': True}, handle)\n        handle.write(\'\\n\')\n        handle.flush()\n        os.fsync(handle.fileno())\n\n\ndef clear_own_normal_completion(token):\n    # Never clear another owner\'s marker or an interrupted/uncertain marker.\n    try: marker = json.loads(BLOCK_PATH.read_text())\n    except (OSError, ValueError): return False\n    if marker.get(\'owner_token\') != token or marker.get(\'phase\') != \'active\': return False\n    BLOCK_PATH.unlink()\n    return True\n\n\ndef run_prepared(**kwargs):\n    require(kwargs.get(\'authorize_gpu\') is True, \'Prepared only: explicit future target authorization required\')\n    require(_DEDICATED_RUNNER, \'GPU workflow must run in dedicated supervisor process, never the notebook kernel\')\n    require(hasattr(os, \'pidfd_open\') and hasattr(signal, \'pidfd_send_signal\'), \'Linux pidfd supervision is required\')\n    # One canonical cross-notebook GPU0 lease; never wait or run concurrently.\n    require(not BLOCK_PATH.exists(), \'GPU0 lane has unresolved cleanup; owner reconciliation is required\')\n    with LOCK_PATH.open(\'a\') as lock:\n        try:\n            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError as exc:\n            raise RuntimeError(\'GPU0 is leased by another Soup notebook/run\') from exc\n        token = uuid.uuid4().hex\n        normal_verified_completion = False\n        try:\n            require(not BLOCK_PATH.exists(), \'GPU0 became blocked before lease acquisition\')\n            create_active_marker(token)\n            outcome = _run_prepared_locked(**kwargs)\n            normal_verified_completion = outcome[1].get(\'clean_process_completion\') is True\n            return outcome\n        except (KeyboardInterrupt, InterruptedError) as exc:\n            dump(BLOCK_PATH, {\'owner_token\': token, \'reason\': type(exc).__name__, \'phase\': \'supervisor interrupted\',\n                              \'manual_owner_reconciliation_required\': True})\n            raise\n        finally:\n            # Also cover failures between supervised commands (e.g. inventory).\n            remaining = OwnedProcesses(None)\n            if remaining.observe():\n                normal_verified_completion = False\n                dump(BLOCK_PATH, {\'owner_token\': token, \'reason\': \'supervisor exit with owned children\', \'cleanup\': \'IN_PROGRESS\',\n                                  \'manual_owner_reconciliation_required\': True})\n                cleanup = remaining.cleanup()\n                dump(BLOCK_PATH, {\'owner_token\': token, \'reason\': \'supervisor exit with owned children\', \'cleanup\': cleanup,\n                                  \'manual_owner_reconciliation_required\': True})\n            if normal_verified_completion:\n                clear_own_normal_completion(token)\n            # Every failure, native timeout, interrupt or abrupt death retains\n            # the marker; only clean owned completion removes its own token.\n            fcntl.flock(lock, fcntl.LOCK_UN)\n\n\ndef _run_prepared_locked(*, authorize_gpu, repo, output_root, hf_hub_cache, sources, hashes,\n                         protected_hashes, timeout_s=900):\n    require(authorize_gpu is True, \'Prepared only: set AUTHORIZE_GPU=True only for an explicitly authorized target run\')\n    require(sys.version_info[:3] == (3, 13, 15), \'Exact Python3.13.15 required; no installation is performed\')\n    versions = {name: importlib.metadata.version(name) for name in PACKAGES}\n    require(versions == PACKAGES, \'Exact direct package pins required; no installation is performed\')\n    require(timeout_s == 900, \'This notebook freezes a 900-second native budget for both sides; new cohort requires review\')\n    require(len(os.sched_getaffinity(0)) == 4, \'Use exactly four assigned logical CPUs; do not assume CPU IDs\')\n    env = dict(os.environ, CUDA_VISIBLE_DEVICES=\'0\', CUDA_DEVICE_ORDER=\'PCI_BUS_ID\',\n               PYTORCH_CUDA_ALLOC_CONF=\'expandable_segments:True\', PYTHONUNBUFFERED=\'1\',\n               HF_HUB_OFFLINE=\'1\', TRANSFORMERS_OFFLINE=\'1\', HF_DATASETS_OFFLINE=\'1\',\n               HF_HUB_CACHE=str(Path(hf_hub_cache).resolve()), TOKENIZERS_PARALLELISM=\'false\')\n    # Inventory is a separate process; the notebook never creates its own CUDA context.\n    inventory_code = \'\'\'import json,subprocess\nactive=subprocess.run(["nvidia-smi","--query-compute-apps=pid,used_gpu_memory","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True)\nassert not active.stdout.strip(), "GPU compute process present before inventory"\nimport torch\nassert torch.cuda.device_count()==1 and torch.cuda.get_device_name(0)=="Tesla T4"\nassert torch.cuda.get_device_capability(0)==(7,5) and torch.version.cuda=="12.8"\np=subprocess.run(["nvidia-smi","--query-gpu=index,memory.used","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True)\nrows=[tuple(int(x.strip()) for x in line.split(",")) for line in p.stdout.splitlines()]\nassert rows and sum(x[1] for x in rows)<1000, "All exposed GPUs must be idle"\nprint(json.dumps({"gpu":"Tesla T4","visible_count":1,"torch":torch.__version__,"torch_git":torch.version.git_version,"external_gpu_memory":rows}))\'\'\'\n    inventory = json.loads(subprocess.check_output([sys.executable, \'-c\', inventory_code], env=env, text=True))\n    cache = public_cache_inventory(hf_hub_cache)\n    root, target = stage(repo, output_root, sources, hashes, protected_hashes)\n    dump(root / \'environment.json\', {\'versions\': versions, \'gpu\': inventory, \'cpu_affinity\': sorted(os.sched_getaffinity(0)),\n                                     \'python\': sys.version, \'public_cache\': cache, \'native_timeout_s\': timeout_s})\n    # Isolated preflight caches are never inherited by measured runs.\n    preenv = dict(env, TRITON_CACHE_DIR=str(root/\'preflight_triton\'), TORCHINDUCTOR_CACHE_DIR=str(root/\'preflight_inductor\'))\n    preflight = command([sys.executable, \'-B\', \'-m\', \'solution.verify_memory\', \'--out\', str(root/\'preflight.json\')], root/\'preflight.log\', target, preenv)\n    dump(root / \'preflight_command.json\', preflight)\n    require(preflight[\'exit_code\'] == 0, \'Target preflight failed; retained evidence at \' + str(root))\n    regressions = command([sys.executable, \'-B\', \'-m\', \'pytest\', \'tests\', \'-q\'], root/\'regressions.log\', target,\n                          dict(env, SOUP_APPLY_SOLUTION=\'1\'))\n    dump(root / \'regression_command.json\', regressions)\n    require(regressions[\'exit_code\'] == 0, \'Unchanged Soup regressions failed; do not measure\')\n    # Preflight/regression children have exited before this fresh idle check.\n    # This short-lived inventory child also exits before measured Soup starts.\n    post_preflight_inventory = json.loads(subprocess.check_output(\n        [sys.executable, \'-c\', inventory_code], env=env, text=True))\n    dump(root/\'idle_after_preflight.json\', post_preflight_inventory)\n    measured = command([sys.executable, \'-B\', \'harness/run_task.py\', \'--repeats\', \'2\', \'--seed\', \'1\',\n                        \'--timeout\', str(timeout_s), \'--out\', str(root/\'runs\')], root/\'harness.log\', target, env)\n    dump(root / \'harness_command.json\', measured)\n    # Read/report partial routes even when the wrapper or native task failed.\n    report = summarize_evidence(root/\'runs/result.json\', root)\n    if measured[\'exit_code\'] != 0:\n        report.update(status=\'FAILED_OR_INCOMPLETE\', score=None, admission=\'UNKNOWN\', ratios=None)\n        report[\'errors\'].append(\'official harness command failed\')\n        dump(root/\'review_summary.json\', report)\n    for name, expected in protected_hashes.items():\n        require(sha(target/name) == expected, \'Protected official file changed during execution: \'+name)\n    report[\'clean_process_completion\'] = False\n    if report[\'status\'] == \'COMPLETE_COMPONENT_MEASUREMENT\':\n        checks = command([sys.executable, \'-B\', \'harness/check_rules.py\', \'--result\', str(root/\'runs/result.json\')], root/\'official_checks.log\', target, env)\n        dump(root/\'checks_command.json\', checks)\n        report[\'clean_process_completion\'] = checks[\'exit_code\'] == 0\n        # A component result is never converted to admission PASS or points.\n    dump(root/\'review_summary.json\', report)\n    print(json.dumps({\'evidence\': str(root), \'status\': report[\'status\'], \'score\': None, \'admission\': \'UNKNOWN\'}, indent=2))\n    return root, report\n\n\ndef main():\n    global _DEDICATED_RUNNER\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--config\', required=True)\n    args = parser.parse_args()\n    config = json.loads(Path(args.config).read_text())\n    require(config.get(\'authorize_gpu\') is True, \'Explicit future GPU authority required\')\n    require(ctypes.CDLL(None, use_errno=True).prctl(36, 1, 0, 0, 0) == 0,\n            \'Cannot establish dedicated Linux child subreaper\')\n    _DEDICATED_RUNNER = True\n    def interrupted(signum, frame):\n        raise InterruptedError(\'Dedicated runner interrupted by signal \' + str(signum))\n    signal.signal(signal.SIGTERM, interrupted)\n    signal.signal(signal.SIGINT, interrupted)\n    run_prepared(**config)\n\n\nif __name__ == \'__main__\': main()\n'


In [ ]:
# The dedicated supervisor, not the notebook, owns CUDA subprocesses/cleanup.
# Default False fails before process launch or GPU use.
assert AUTHORIZE_GPU is True, "Prepared only: explicit target-run authorization required"
import os, signal, subprocess, sys, tempfile
from pathlib import Path
assert hasattr(os, "pidfd_open") and hasattr(signal, "pidfd_send_signal"), "Linux pidfd support required"
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
launch_dir = Path(tempfile.mkdtemp(prefix="memory_vs1_launcher_", dir=OUTPUT_ROOT))
(launch_dir / "runner.py").write_text(RUNNER_SOURCE)
settings = dict(authorize_gpu=True, repo=OFFICIAL_REPO, output_root=OUTPUT_ROOT,
                hf_hub_cache=HF_HUB_CACHE, sources=SOURCES, hashes=SOURCE_HASHES,
                protected_hashes=PROTECTED_HASHES, timeout_s=900)
(launch_dir / "config.json").write_text(json.dumps(settings))
proc = subprocess.Popen([sys.executable, str(launch_dir / "runner.py"),
                         "--config", str(launch_dir / "config.json")], start_new_session=True)
pidfd = os.pidfd_open(proc.pid, 0)
try:
    try:
        code = proc.wait()
    except BaseException:
        Path("/tmp/soup-gpu0.lock.blocked").write_text(json.dumps({
            "reason": "notebook launcher interrupted", "supervisor_pid": proc.pid,
            "manual_owner_reconciliation_required": True}))
        try: signal.pidfd_send_signal(pidfd, signal.SIGTERM, None, 0)
        except ProcessLookupError: pass
        # Keep waiting for owned cleanup; another Ctrl-C must not release it.
        while proc.poll() is None:
            try: proc.wait(timeout=1)
            except (subprocess.TimeoutExpired, KeyboardInterrupt): pass
        raise
    if code < 0:
        Path("/tmp/soup-gpu0.lock.blocked").write_text(json.dumps({
            "reason": "supervisor terminated by signal", "exit_code": code,
            "manual_owner_reconciliation_required": True}))
    assert code == 0, "Supervisor failed; preserve all logs and quarantine; no score/admission"
finally:
    os.close(pidfd)
print("Evidence is under", OUTPUT_ROOT, "— inspect the new memory_vs1_* directory")


## Read the evidence, not a predicted score
`review_summary.json` reports completeness and recovered activation records. `runs/result.json` retains native complete wall time, external nvidia-smi peak, public evaluation and measured own-kernel CUDA events after two warmup steps. A killed/partial run cannot earn ratios, score or admission. Official checks are retained separately only after complete measurements. GCP needs its own baseline; official Kaggle admission remains separate. Do not treat CPU witnesses, logical component bytes or historical trial005 memory as target validation.
